# Week 10 — LLM-mediated interpersonal messages

**Research task:** Hold a broken commitment fixed while changing the writing instruction, relationship and assistance-disclosure setting. Inspect generated writing without claiming to observe sincerity or relationship repair.

**Python introduced:** default parameter values, Boolean values, keyword arguments and `enumerate()`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cjbarrie/GenAI_Soc2026/blob/main/workbook/session10/session10_relationship_mediation.ipynb)

## Prepare the notebook environment

Run this cell first. Colab installs the two SDKs and uses OpenRouter. Local JupyterLab checks the course environment and uses Ollama by default.

In [ ]:
SESSION = "session10"

# Run this cell first. It prepares Colab or checks the local Python environment.
import importlib as setup_importlib
import importlib.util as setup_importlib_util
import os as setup_os
import subprocess as setup_subprocess
import sys as setup_sys
from pathlib import Path as SetupPath

try:
    import google.colab as setup_colab  # type: ignore[import-not-found]
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

course_packages = {
    "openrouter": "openrouter>=0.6,<1",
    "ollama": "ollama>=0.6,<1",
}
if SESSION == "session05":
    course_packages["plotly"] = "plotly>=6.3,<7"
    course_packages["umap"] = "umap-learn>=0.5.9,<1"
    course_packages["pandas"] = "pandas>=2.3,<3"
if SESSION == "session13":
    course_packages["pandas"] = "pandas>=2.2,<3"

missing_packages = [
    package_name
    for package_name in course_packages
    if setup_importlib_util.find_spec(package_name) is None
]

if IN_COLAB:
    if missing_packages:
        packages_to_install = [course_packages[name] for name in missing_packages]
        setup_subprocess.run(
            [
                setup_sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                "--disable-pip-version-check",
                *packages_to_install,
            ],
            check=True,
        )
        setup_importlib.invalidate_caches()

    COURSE_ROOT = SetupPath(
        setup_os.getenv("COURSE_COLAB_ROOT", "/content/GenAI_Soc2026")
    )
    if not COURSE_ROOT.exists():
        setup_subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/cjbarrie/GenAI_Soc2026.git",
                str(COURSE_ROOT),
            ],
            check=True,
        )
    setup_os.chdir(COURSE_ROOT / "workbook" / SESSION)
else:
    COURSE_ROOT = SetupPath.cwd()
    while not (COURSE_ROOT / "config" / "course_models.json").exists() and COURSE_ROOT != COURSE_ROOT.parent:
        COURSE_ROOT = COURSE_ROOT.parent
    if missing_packages:
        missing_text = ", ".join(missing_packages)
        raise ModuleNotFoundError(
            f"This notebook is using a Python environment without: {missing_text}.\n\n"
            "Close Jupyter. Open a terminal in the GenAI_Soc2026 repository and run:\n"
            "    uv sync\n"
            "    uv run jupyter lab\n\n"
            "In VS Code, select the Python interpreter inside the repository's .venv folder."
        )
    if not (COURSE_ROOT / "config" / "course_models.json").exists():
        raise FileNotFoundError(
            "The complete GenAI_Soc2026 repository could not be found. A notebook "
            "downloaded by itself is not enough for local work. Download or clone the "
            "repository, open a terminal in that folder, and run: uv run jupyter lab"
        )

course_root_text = str(COURSE_ROOT)
if course_root_text not in setup_sys.path:
    setup_sys.path.insert(0, course_root_text)

still_missing = [
    package_name
    for package_name in course_packages
    if setup_importlib_util.find_spec(package_name) is None
]
if still_missing:
    raise ModuleNotFoundError(
        "Setup did not make these packages available: " + ", ".join(still_missing)
    )

print("Environment:", "Google Colab" if IN_COLAB else "local course environment")
print("Python executable:", setup_sys.executable)
print("Course root:", COURSE_ROOT)
print("Working folder:", SetupPath.cwd())
print("Python SDKs: ready")
if IN_COLAB:
    print("Route for this runtime: OpenRouter")
    print("Ollama work: complete later in local JupyterLab or on the in-class machine")
else:
    import json as setup_json
    import ollama as setup_ollama

    setup_config = setup_json.loads(
        (COURSE_ROOT / "config" / "course_models.json").read_text()
    )
    setup_local_model = setup_config["local"]["model"]
    try:
        setup_models = setup_ollama.list().models
        setup_model_names = [
            getattr(item, "model", None) or getattr(item, "name", None)
            for item in setup_models
        ]
        print("Ollama server: reachable at localhost:11434")
        if setup_local_model in setup_model_names:
            print("Course local model: ready —", setup_local_model)
        else:
            print("Course local model: NOT INSTALLED —", setup_local_model)
            print("NEXT STEP: open a terminal and run: ollama pull " + setup_local_model)
    except Exception as setup_error:
        print("Ollama server: NOT REACHABLE")
        print("NEXT STEP: start the Ollama application, then run: ollama list")
        print("Diagnostic:", str(setup_error).splitlines()[0])


## Import tools and store the course models

This cell locates the complete repository and selects the hosted and local models. Colab uses OpenRouter; local JupyterLab uses Ollama by default.

In [ ]:
import json
import os
from getpass import getpass
from pathlib import Path

try:
    import ollama
    from openrouter import OpenRouter
except ModuleNotFoundError:
    ollama = None
    OpenRouter = None

ROOT = Path.cwd()
while not (ROOT / "config" / "course_models.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "config" / "course_models.json").exists():
    raise FileNotFoundError("Run this file from inside the complete GenAI_Soc2026 folder.")

config = json.loads((ROOT / "config" / "course_models.json").read_text())
OPENROUTER_MODEL = config["hosted"]["model"]
OLLAMA_MODEL = config["local"]["model"]
HOSTED_MODEL = OPENROUTER_MODEL
LOCAL_MODEL = OLLAMA_MODEL
USE_RECORDED = os.getenv("COURSE_USE_RECORDED", "0") == "1"
ROUTE = "openrouter" if IN_COLAB else "ollama"

## Store the fixed facts and two instructions

The facts dictionary stays unchanged. The two strings allocate different work to the model. Recorded returns provide an outage fallback, not evidence about people.

In [ ]:
facts = {
    "commitment": "help with a move at 8 a.m. Saturday",
    "work_conflict_known": "noon Friday",
    "message_sent": "9 p.m. Friday",
    "reliance": "the recipient declined paid help because of the promise",
    "repairs": ["pay half the cost of a last-minute mover", "help on Sunday"],
}

thin_instruction = (
    "Write a message telling this person that I cannot help them move tomorrow. "
    "Use only the supplied facts and do not invent facts."
)
obligation_instruction = (
    "State the commitment that was broken, acknowledge the late notice and the "
    "plan made in reliance on the promise, explain only the supplied facts, and "
    "offer both available forms of repair. Do not invent feelings, relationship "
    "history or motives."
)

# Recorded outputs demonstrate the code path during an outage. They are not
# observations of human communication or relationship outcomes.
recorded_returns = {
    ("thin", "a close friend of ten years", False): (
        "I’m sorry, but I can’t help with the move tomorrow because I have to work. "
        "I can help on Sunday instead or pay half the cost of a mover."
    ),
    ("obligation", "a colleague from my team", False): (
        "I need to apologize for breaking my commitment to help at 8 tomorrow and "
        "for waiting until 9 tonight to tell you. I learned at noon that I must work. "
        "You declined paid help because you expected me, so I can either pay half the "
        "cost of a last-minute mover or help on Sunday."
    ),
    ("obligation", "a close friend of ten years", False): (
        "I’m sorry. I promised to help at 8 tomorrow, and I should have told you at "
        "noon when I learned I had to work. You turned down paid help because you were "
        "counting on me. I can pay half the cost of a mover or help on Sunday."
    ),
    ("obligation", "my romantic partner", False): (
        "I broke my promise to help at 8 tomorrow, and I waited too long to tell you. "
        "You declined paid help because you relied on me. I can pay half the cost of a "
        "last-minute mover or help on Sunday."
    ),
    ("obligation", "a close friend of ten years", True): (
        "I’m sorry. I promised to help at 8 tomorrow, and I should have told you at "
        "noon when I learned I had to work. You turned down paid help because you were "
        "counting on me. I can pay half the cost of a mover or help on Sunday. I used "
        "an AI writing assistant to help me draft this message."
    ),
}

## Define a function with a default Boolean parameter

The first three parameters require values. `disclose_assistance=False` supplies a default Boolean when the caller omits that keyword argument.

In [ ]:
def draft_message(
    relationship,
    instruction_name,
    route,
    disclose_assistance=False,
):
    """Return a record containing one model-generated message and its inputs."""
    if instruction_name == "thin":
        instruction = thin_instruction
    elif instruction_name == "obligation":
        instruction = obligation_instruction
    else:
        raise ValueError("instruction_name must be 'thin' or 'obligation'")

    disclosure_request = ""
    if disclose_assistance:
        disclosure_request = (
            "\nEnd with this factual disclosure: I used an AI writing assistant "
            "to help me draft this message."
        )

    prompt = f"""{instruction}

Relationship: {relationship}

Facts:
{json.dumps(facts, indent=2)}
{disclosure_request}

Return only the message that could be sent."""

    print("relationship:", relationship, type(relationship))
    print("instruction_name:", instruction_name, type(instruction_name))
    print("disclose_assistance:", disclose_assistance, type(disclose_assistance))
    print("route:", route, type(route))
    print("Prompt sent to the model:\n", prompt)

    if USE_RECORDED:
        key = (instruction_name, relationship, disclose_assistance)
        try:
            raw_output = recorded_returns[key]
        except KeyError as error:
            raise KeyError(
                "No recorded return exists for that combination. Use a live route or "
                "restore one of the supplied combinations."
            ) from error
        model_name = "recorded-teaching-return"
    elif route == "openrouter":
        if OpenRouter is None:
            raise ModuleNotFoundError("Run 'uv sync --frozen' before using OpenRouter.")
        api_key = os.getenv("OPENROUTER_API_KEY") or getpass("OpenRouter course key (hidden): ")
        with OpenRouter(api_key=api_key) as client:
            response = client.chat.send(
                model=OPENROUTER_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
            )
        raw_output = response.choices[0].message.content
        model_name = OPENROUTER_MODEL
    elif route == "ollama":
        if ollama is None:
            raise ModuleNotFoundError("Run 'uv sync --frozen' before using Ollama.")
        response = ollama.chat(think=False,
            model=OLLAMA_MODEL,
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": 0},
        )
        raw_output = response.message.content
        model_name = OLLAMA_MODEL
    else:
        raise ValueError("route must be 'openrouter' or 'ollama'")

    record = {
        "relationship": relationship,
        "instruction_name": instruction_name,
        "disclose_assistance": disclose_assistance,
        "route": "recorded" if USE_RECORDED else route,
        "model": model_name,
        "facts": facts.copy(),
        "raw_message": raw_output,
    }
    print("Raw model return:\n", raw_output)
    print("Function output type:", type(record))
    return record

## Compare two instructions

Both calls use the same relationship and facts. Keyword arguments make each input visible at the call site.

In [ ]:
thin_record = draft_message(
    relationship="a close friend of ten years",
    instruction_name="thin",
    route=ROUTE,
)
obligation_record = draft_message(
    relationship="a close friend of ten years",
    instruction_name="obligation",
    route=ROUTE,
)
print("\nTHIN INSTRUCTION\n", thin_record["raw_message"])
print("\nRELATIONAL-OBLIGATION INSTRUCTION\n", obligation_record["raw_message"])

## Number the relationship conditions

`enumerate(..., start=1)` returns a condition number and a relationship label on each iteration. This first loop only prints those two values so that they remain visible.

In [ ]:
relationships = [
    "a colleague from my team",
    "a close friend of ten years",
    "my romantic partner",
]
for position, relationship in enumerate(relationships, start=1):
    print(position, relationship)

## Repeat one instruction across relationships

The second loop calls the function for each relationship. It appends one record dictionary at a time to the initially empty list.

In [ ]:
relationship_records = []
for position, relationship in enumerate(relationships, start=1):
    print("\nCondition", position, "of", len(relationships))
    record = draft_message(
        relationship=relationship,
        instruction_name="obligation",
        route=ROUTE,
    )
    relationship_records.append(record)

## Make disclosure an explicit input

The final keyword argument changes from its default `False` to `True`. The resulting text remains a generated message, not evidence of a recipient response.

In [ ]:
undisclosed_record = obligation_record
disclosed_record = draft_message(
    relationship="a close friend of ten years",
    instruction_name="obligation",
    route=ROUTE,
    disclose_assistance=True,
)
print("\nWITHOUT DISCLOSURE REQUEST\n", undisclosed_record["raw_message"])
print("\nWITH DISCLOSURE REQUEST\n", disclosed_record["raw_message"])

## Keep researcher notes separate

Write observations after inspecting the text. Do not ask the model to score sincerity, appropriateness or relationship quality.

In [ ]:
researcher_notes = {
    "instruction_comparison": "",
    "relationship_comparison": "",
    "disclosure_comparison": "",
    "invented_facts": [],
    "human_evidence_needed": (
        "Recipient interpretations, sender experience and later interaction data."
    ),
}
print(json.dumps(researcher_notes, indent=2))

# ONE CHANGE: edit one relationship description or set disclose_assistance=True
# for another relationship. Do not treat the generated difference as a human effect.

## Your change and recording

Change one relationship description or set `disclose_assistance=True` for another relationship. Explain the default value, the keyword argument, the first `enumerate()` iteration and the returned record. State which human outcomes remain unobserved.